In [6]:
import numpy as np
import matplotlib.pyplot as plt
import csv
from mpl_toolkits.mplot3d import Axes3D

In [7]:
class CombinedFeatures:
    def __init__(self, r):
        # Initialize instance variables by converting each value from the dictionary to a float
        self.kb = float(r['kb'])
        self.g_Na = float(r['g_Na'])
        self.g_Cl = float(r['g_Cl'])
        self.g_K = float(r['g_K'])
        self.Num_burst = float(r['Num_burst'])
        self.INF_COND = float(r['INF_COND'])
        self.Num_Spike_1 = float(r['Num_Spike_1'])
        self.Baseline_b_1 = float(r['Baseline_b_1'])
        self.Baseline_a_1 = float(r['Baseline_a_1'])
        
        # Additional baseline and amplitude values for category 1
        self.Baseline_jump_on_1 = float(r['Baseline_jump_on_1'])
        self.Baseline_jump_off_1 = float(r['Baseline_jump_off_1'])
        self.amplitude_ratio_on_1 = float(r['amplitude_ratio_on_1'])
        self.amplitude_ratio_off_1 = float(r['amplitude_ratio_off_1'])
        self.ave_amp_1 = float(r['ave_amp_1'])
        self.var_amp_1 = float(r['var_amp_1'])
        self.start_1 = float(r['start_1'])
        self.end_1 = float(r['end_1'])
        self.ave_min_1 = float(r['ave_min_1'])
        self.var_min_1 = float(r['var_min_1'])
        self.starting_T_1 = float(r['starting_T_1'])
        self.ending_T_1 = float(r['ending_T_1'])
        self.starting_A_1 = float(r['starting_A_1'])
        self.ending_A_1 = float(r['ending_A_1'])

        # Similar metrics for category 2
        self.Num_Spike_2 = float(r['Num_Spike_2'])
        self.Baseline_b_2 = float(r['Baseline_b_2'])
        self.Baseline_a_2 = float(r['Baseline_a_2'])
        self.Baseline_jump_on_2 = float(r['Baseline_jump_on_2'])
        self.Baseline_jump_off_2 = float(r['Baseline_jump_off_2'])
        self.amplitude_ratio_on_2 = float(r['amplitude_ratio_on_2'])
        self.amplitude_ratio_off_2 = float(r['amplitude_ratio_off_2'])
        self.ave_amp_2 = float(r['ave_amp_2'])
        self.var_amp_2 = float(r['var_amp_2'])
        self.start_2 = float(r['start_2'])
        self.end_2 = float(r['end_2'])
        self.ave_min_2 = float(r['ave_min_2'])
        self.var_min_2 = float(r['var_min_2'])
        self.starting_T_2 = float(r['starting_T_2'])
        self.ending_T_2 = float(r['ending_T_2'])
        self.starting_A_2 = float(r['starting_A_2'])
        self.ending_A_2 = float(r['ending_A_2'])
        
        # Category for the data entry
        self.Category = r['Category']


In [8]:
def load_combined_features(file_path, delimiter='\t'):
    instances = []
    with open(file_path, 'r', newline='') as f:
        reader = csv.DictReader(f, delimiter=delimiter)
        for row in reader:
            try:
                instance = CombinedFeatures(row)
                instances.append(instance)
            except KeyError as e:
                print(f"Missing key in file: {e} in row: {row}")
    return instances

In [9]:
file =  "/Users/gianmarcocafaro/Desktop/cat_40/3ritest_merged_cat.txt" 
Data = load_combined_features(file)

## `first_selection` Categorization

This code snippet categorizes instances from the `Data` list into three categories: `'SupH'`, `'not'`, and `'no signal'`. It uses conditions based on several attributes of the `CombinedFeatures` class, such as `Num_Spike_1`, `amplitude_ratio_on_1`, `amplitude_ratio_on_2`, `ave_min_1`, `ave_amp_1`, and `Num_Spike_1` to determine which category each data point belongs to.

### Categories:
- **'SupH'**: 
    - Entries with `amplitude_ratio_on_1` less than 0.8.
    - Entries where `amplitude_ratio_on_2` is less than 0.8 and the sum of `ave_min_1` and `ave_amp_1` is less than 6, with `Num_Spike_1` being less than 260.
- **'not'**:
    - Entries that do not meet any of the conditions for 'SupH' or 'no signal'.
- **'no signal'**:
    - Entries where `Num_Spike_1` is zero.


In [10]:
# Create a dictionary to store selections based on different categories
first_selection = {}
first_selection['SupH'] = []  # Category for 'SupH' type entries
first_selection['not'] = []    # Category for 'not' type entries
first_selection['no signal'] = []  # Category for entries with 'no signal'

# Loop through the 'Data' list, which contains instances of 'CombinedFeatures'
for t in Data:
    if t.Num_Spike_1 == 0:  # If the number of spikes is zero, classify as 'no signal'
        first_selection['no signal'].append(t)
    elif t.amplitude_ratio_on_1 < 0.8:  # If the amplitude ratio on channel 1 is less than 0.8, classify as 'SupH'
        first_selection['SupH'].append(t)
    elif t.amplitude_ratio_on_2 < 0.8:  # If the amplitude ratio on channel 2 is less than 0.8, check additional conditions
        if t.ave_min_1 + t.ave_amp_1 < 6 and t.Num_Spike_1 < 260:  # If average amplitude and number of spikes meet conditions, classify as 'SupH'
            first_selection['SupH'].append(t)
        else:  # Otherwise, classify as 'not'
            first_selection['not'].append(t)
    else:  # If none of the above conditions match, classify as 'not'
        first_selection['not'].append(t)


## `second_selection_SupH` Categorization

This code snippet further categorizes entries from the `'SupH'` category, which was previously defined in `first_selection`. The instances are classified into three sub-categories: `'SN-SupH'`, `'SN-SupH | Sup H - Infinity'`, and `'SN-SupH | Sup H - SH'`. The categorization is based on the attributes of `Num_Spike_1`, `Num_burst`, and `var_amp_1` in the `CombinedFeatures` class.

### Categories:
- **'SN-SupH'**:
    - Entries where `Num_burst` equals 1.
    - Entries where `Num_burst` equals 2 and `var_amp_1` is less than 20.
- **'SN-SupH | Sup H - Infinity'**:
    - Entries where `Num_Spike_1` is greater than or equal to 5000.
- **'SN-SupH | Sup H - SH'**:
    - Entries that do not meet the conditions for 'SN-SupH' or 'SN-SupH | Sup H - Infinity'.


In [11]:
# Create a dictionary to store further classifications of the 'SupH' category
second_selection_SupH = {}
second_selection_SupH['SN-SupH'] = []  # Category for 'SN-SupH' type entries
second_selection_SupH['SN-SupH | Sup H - Infinity'] = []  # Category for 'SupH' with Num_Spike_1 >= 5000
second_selection_SupH['SN-SupH | Sup H - SH'] = []  # Category for 'SupH' with special conditions

# Loop through the 'SupH' category from the first selection
for t in first_selection['SupH']:
    if t.Num_Spike_1 >= 5000:  # If the number of spikes is greater than or equal to 5000, classify as 'SN-SupH | Sup H - Infinity'
        second_selection_SupH['SN-SupH | Sup H - Infinity'].append(t)
    elif t.Num_burst == 1:  # If there is only 1 burst, classify as 'SN-SupH'
        second_selection_SupH['SN-SupH'].append(t)
    elif t.Num_burst == 2 and t.var_amp_1 < 20:  # If there are 2 bursts and the variance in amplitude is less than 20, classify as 'SN-SupH'
        second_selection_SupH['SN-SupH'].append(t)
    else:  # If no other condition is met, classify as 'SN-SupH | Sup H - SH'
        second_selection_SupH['SN-SupH | Sup H - SH'].append(t)


## `second_selection_not` Categorization

This code snippet further categorizes entries from the `'not'` category, which was previously defined in `first_selection`. The instances are classified into five sub-categories: `'SNIC-SNIC'`, `'SNIC-Infinity'`, `'SN-SH'`, `'SNIC-SH'`, and `'SN-Infinity'`. The categorization is based on the attributes of `Num_Spike_2`, `Num_Spike_1`, `Baseline_jump_on_1`, `var_min_1`, and `starting_T_1`.

### Categories:
- **'SNIC-SNIC'**:
    - Entries where `Baseline_jump_on_1` is less than 0 and `starting_T_1` is greater than 1.
- **'SNIC-Infinity'**:
    - Entries where the ratio of `Num_Spike_2` to `Num_Spike_1` is less than 1, and `Baseline_jump_on_1` is less than 0, `var_min_1` is less than 0.1, and `starting_T_1` is greater than 1.
- **'SN-SH'**:
    - Entries that do not meet the conditions for `'SNIC-SNIC'`, `'SNIC-Infinity'`, or `'SN-Infinity'`.
- **'SN-Infinity'**:
    - Entries where the ratio of `Num_Spike_2` to `Num_Spike_1` is less than 1, and the conditions for `'SNIC-Infinity'` are not met.



In [12]:
second_selection_not = {}
second_selection_not['SNIC-SNIC'] = []
second_selection_not['SNIC-Infinity'] = []
second_selection_not['SN-SH'] = []
second_selection_not['SNIC-SH'] = []
second_selection_not['SN-Infinity'] = []

for t in first_selection['not']:
    if t.Num_Spike_2/t.Num_Spike_1 < 1:
        if t.Baseline_jump_on_1 < 0 and t.var_min_1 < 0.1 and t.starting_T_1 > 1:
            second_selection_not['SNIC-Infinity'].append(t)
        else:
            second_selection_not['SN-Infinity'].append(t)
    elif t.Baseline_jump_on_1 < 0 and t.starting_T_1 > 1:
        second_selection_not['SNIC-SNIC'].append(t)
    else:
        second_selection_not['SN-SH'].append(t)

## `writer(path)` Function

The `writer(path)` function is responsible for writing data from various categories into a tab-delimited text file at the given `path`.

### Function Flow:
1. **Headers Setup**:
   - The function defines a list of headers, representing the attributes of the dataset, such as `'kb'`, `'g_Na'`, `'Num_Spike_1'`, and others.
   
2. **Opening the File**:
   - The file located at `path` is opened in write mode. The headers are written as the first line.

3. **Data Writing**:
   - The `write_data()` function is responsible for writing the actual data to the file.
   - For each data selection (such as `'no signal'`, `'SNIC-Infinity'`), `write_data()` is called to write out the data rows for each entry.
   - The category name (such as `'no signal'`, `'SNIC-Infinity'`) is appended to the end of each data row to indicate which category the data belongs to.


In [13]:
def writer(path):
    headers = [
        'kb', 'g_Na', 'g_Cl', 'g_K', 'Num_burst', 'INF_COND', 'Num_Spike_1',
        'Baseline_b_1', 'Baseline_a_1', 'Baseline_jump_on_1', 'Baseline_jump_off_1',
        'amplitude_ratio_on_1', 'amplitude_ratio_off_1', 'ave_amp_1', 'var_amp_1',
        'start_1', 'end_1', 'ave_min_1', 'var_min_1', 'starting_T_1', 'ending_T_1',
        'starting_A_1', 'ending_A_1', 'Num_Spike_2', 'Baseline_b_2', 'Baseline_a_2',
        'Baseline_jump_on_2', 'Baseline_jump_off_2', 'amplitude_ratio_on_2',
        'amplitude_ratio_off_2', 'ave_amp_2', 'var_amp_2', 'start_2', 'end_2',
        'ave_min_2', 'var_min_2', 'starting_T_2', 'ending_T_2', 'starting_A_2', 'ending_A_2',
        'Category'
    ]
    
    with open(path, 'w') as t:
        # Write headers
        t.write('\t'.join(headers) + '\n')
        
        # Write the data based on different categories
        def write_data(selection, category):
            for i in selection:
                # Collect values for each attribute and add the category
                values = [str(getattr(i, attr)) for attr in headers[:-1]]  # Get all attributes except 'Category'
                values.append(category)  # Append the category at the end
                t.write('\t'.join(values) + '\n')
        
        # Write data for each category
        write_data(first_selection['no signal'], 'no signal')
        write_data(second_selection_not['SNIC-Infinity'], 'SNIC-Infinity')
        write_data(second_selection_not['SN-Infinity'], 'SN-Infinity')
        write_data(second_selection_not['SNIC-SNIC'], 'SNIC-SNIC')
        write_data(second_selection_not['SN-SH'], 'SN-SH')
        write_data(second_selection_SupH['SN-SupH | Sup H - Infinity'],'SN-SupH | Sup H - Infinity')
        write_data(second_selection_SupH['SN-SupH | Sup H - SH'],'SN-SupH | Sup H - SH')
        write_data(second_selection_SupH['SN-SupH'],'SN-SupH')